In [ ]:
# ========== 0) Colab Drive 마운트 ==========
from google.colab import drive
drive.mount('/content/drive')

# ========== 1) 설정 ==========
import os, glob, unicodedata
import pandas as pd
from transformers import pipeline
import torch

# 데이터 폴더(Drive)
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"

# 체크포인트/결과 저장 폴더(Drive)
SAVE_DIR = os.path.join(DATA_DIR, "_ckpt")  # <- 드라이브 안에 명시적으로 저장
os.makedirs(SAVE_DIR, exist_ok=True)

CKPT_PRED_FILE    = os.path.join(SAVE_DIR, "sentiment_ckpt_byfile.csv")  # 배치 예측 저장
STORE_SCORES_FILE = os.path.join(SAVE_DIR, "store_scores.csv")           # 최종 매장 점수
BATCH_SIZE        = 32
COUNT_BASE        = 50  # count 정규화 기준

# 지역 목록(NFC)
METRO_MAP = ["서울","부산","대구","인천","광주","대전","울산","세종","경기",
             "강원","충북","충남","전북","전남","경북","경남","제주"]
NFC_METRO = {unicodedata.normalize("NFC", m): m for m in METRO_MAP}

print(f"[INFO] DATA_DIR={DATA_DIR}")
print(f"[INFO] SAVE_DIR={SAVE_DIR}")
print(f"[INFO] CKPT_PRED_FILE={CKPT_PRED_FILE}")
print(f"[INFO] STORE_SCORES_FILE={STORE_SCORES_FILE}")

# ========== 2) 유틸 ==========
def list_region_files():
    """DATA_DIR에 있는 *_filtered.csv 중 지역명 매칭되는 파일만 반환."""
    files = glob.glob(os.path.join(DATA_DIR, "*_filtered.csv"))
    items = []
    for f in files:
        name = os.path.basename(f).replace("_filtered.csv", "")
        n_name = unicodedata.normalize("NFC", name)
        if n_name in NFC_METRO:
            items.append((f, NFC_METRO[n_name]))  # (path, region)
    return items

def load_ckpt():
    if os.path.exists(CKPT_PRED_FILE):
        return pd.read_csv(CKPT_PRED_FILE)
    return pd.DataFrame(columns=["source_file","row_id","sentiment_score","class_prob"])

def append_ckpt(df_rows: pd.DataFrame):
    """체크포인트 파일에 배치 예측 결과를 즉시 append."""
    if df_rows is None or df_rows.empty:
        return
    header = not os.path.exists(CKPT_PRED_FILE)
    df_rows.to_csv(CKPT_PRED_FILE, mode="a", header=header, index=False)
    # 확인용 로그
    try:
        size = os.path.getsize(CKPT_PRED_FILE)
        print(f"[CKPT] wrote {len(df_rows)} rows -> {CKPT_PRED_FILE} (size={size} bytes)")
    except Exception as e:
        print("[CKPT] size check failed:", e)

def build_pipeline():
    device = 0 if torch.cuda.is_available() else -1
    print(f"[INFO] torch.cuda.is_available()={torch.cuda.is_available()} -> device={device}")
    return pipeline(
        "sentiment-analysis",
        model="nlptown/bert-base-multilingual-uncased-sentiment",
        device=device,
        batch_size=BATCH_SIZE,
        truncation=True
    )

# ========== 3) 새로 추가된 파일만 예측 (체크포인트 생성/추가) ==========
def run_predictions_on_new_files():
    ckpt = load_ckpt()
    done_files = set(ckpt["source_file"].unique())
    region_files = list_region_files()
    if not region_files:
        raise FileNotFoundError(f"No *_filtered.csv in {DATA_DIR}")

    # 이번에 새로 추가된 파일만 처리
    targets = []
    for fpath, region in region_files:
        bname = os.path.basename(fpath)
        if bname not in done_files:
            targets.append((fpath, region))

    if not targets:
        print("[INFO] 새로 처리할 파일 없음. 체크포인트 재사용.")
        return []

    pipe = build_pipeline()
    processed_regions = []

    for fpath, region in targets:
        bname = os.path.basename(fpath)
        print(f"[RUN] {bname} ({region})")
        df = pd.read_csv(fpath)
        if "review_text" not in df.columns:
            print(f"  - Warning: review_text 없음 → skip")
            continue

        df = df.reset_index(drop=True)
        texts = df["review_text"].astype(str).tolist()
        n = len(texts)

        # 배치별 즉시 append (중간 중단 대비)
        for start in range(0, n, BATCH_SIZE):
            end = min(start + BATCH_SIZE, n)
            batch_texts = texts[start:end]
            outs = pipe(batch_texts)
            rows = []
            for i, out in enumerate(outs, start):
                star = int(out["label"][0])     # '4 stars' -> 4
                sent = (star - 1) / 4          # 0~1
                prob = out["score"]            # 해당 레이블 확신도
                rows.append([bname, i, sent, prob])
            append_ckpt(pd.DataFrame(rows, columns=["source_file","row_id","sentiment_score","class_prob"]))

        processed_regions.append(region)

    return processed_regions

# ========== 4) 점수 집계 ==========
def build_scores_for_regions(target_regions=None):
    """체크포인트와 원본을 합쳐 (region, store_name, address)별 점수 산출."""
    ckpt = load_ckpt()
    if ckpt.empty:
        raise RuntimeError("체크포인트가 비어 있습니다. 먼저 예측을 실행하세요.")

    region_files = list_region_files()
    parts = []
    for fpath, region in region_files:
        if target_regions is not None and region not in target_regions:
            continue
        bname = os.path.basename(fpath)
        df = pd.read_csv(fpath).reset_index().rename(columns={"index":"row_id"})
        df["source_file"] = bname
        keep = ["source_file","row_id","store_name","address"]
        if "review_text" in df.columns:  # 있으면 유지
            keep.append("review_text")
        df = df[keep]
        merged = df.merge(ckpt, on=["source_file","row_id"], how="inner")
        if merged.empty:
            print(f"[WARN] {bname} merge 결과 없음(체크포인트 부족)")
            continue
        merged["region"] = region
        parts.append(merged)

    if not parts:
        raise RuntimeError("병합 결과가 없습니다. target_regions를 확인하세요.")
    combined = pd.concat(parts, ignore_index=True)

    combined["count"] = 1
    agg = combined.groupby(["region","store_name","address"], dropna=False).agg(
        count          = ("count","sum"),
        avg_sentiment  = ("sentiment_score","mean"),
        avg_class_prob = ("class_prob","mean"),
    ).reset_index()

    agg["count_norm"]   = (agg["count"] / COUNT_BASE).clip(0, 1)
    agg["review_score"] = agg[["count_norm","avg_sentiment","avg_class_prob"]].mean(axis=1)
    return agg

# ========== 5) 기존 점수 파일에 새 지역만 갱신 병합 (반환 추가) ==========
def update_store_scores(new_regions, return_df=False):
    if not new_regions:
        print("[INFO] 갱신할 지역 없음")
        if return_df and os.path.exists(STORE_SCORES_FILE):
            return pd.read_csv(STORE_SCORES_FILE)
        return None

    new_scores = build_scores_for_regions(target_regions=new_regions)

    if os.path.exists(STORE_SCORES_FILE):
        old = pd.read_csv(STORE_SCORES_FILE)
        updated = pd.concat([
            old[~old["region"].isin(new_scores["region"].unique())],
            new_scores
        ], ignore_index=True)
    else:
        updated = new_scores

    updated.to_csv(STORE_SCORES_FILE, index=False)
    print(f"[DONE] {STORE_SCORES_FILE} 갱신 (regions={sorted(set(new_regions))})")

    if return_df:
        return updated

# ========== 6) 실행 & 최종 DF 확인 ==========
processed_regions = run_predictions_on_new_files()   # 새 파일만 예측 수행
updated = update_store_scores(processed_regions, return_df=True)

print("\n[RESULT] 최종 store_scores 샘플 (상위 20):")
if isinstance(updated, pd.DataFrame):
    # 이번에 갱신된 전체 DF 메모리에서 바로 확인
    print(updated.sort_values(["region","review_score"], ascending=[True,False]).head(20).to_string(index=False))
else:
    # 갱신된 DF를 못 받았으면 저장된 CSV를 읽어서 확인
    if os.path.exists(STORE_SCORES_FILE):
        df_final = pd.read_csv(STORE_SCORES_FILE)
        print(df_final.sort_values(["region","review_score"], ascending=[True,False]).head(20).to_string(index=False))
    else:
        print("store_scores.csv가 아직 존재하지 않습니다.")

Mounted at /content/drive
[INFO] DATA_DIR=/content/drive/MyDrive/Colab Notebooks
[INFO] SAVE_DIR=/content/drive/MyDrive/Colab Notebooks/_ckpt
[INFO] CKPT_PRED_FILE=/content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv
[INFO] STORE_SCORES_FILE=/content/drive/MyDrive/Colab Notebooks/_ckpt/store_scores.csv
[INFO] torch.cuda.is_available()=True -> device=0


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


[RUN] 광주_filtered.csv (광주)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2898021 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2899813 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2901612 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2903435 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2905259 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2907083 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2908911 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2910734 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyD

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2914389 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2916214 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2918047 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2919866 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2921697 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2923518 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2925343 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sentiment_ckpt_byfile.csv (size=2927167 bytes)
[CKPT] wrote 32 rows -> /content/drive/MyDrive/Colab Notebooks/_ckpt/sen